# Implementing Recommender Systems - Lab

## Introduction

In this lab, you'll practice creating a recommender system model using `surprise`. You'll also get the chance to create a more complete recommender system pipeline to obtain the top recommendations for a specific user.


## Objectives

In this lab you will: 

- Use surprise's built-in reader class to process data to work with recommender algorithms 
- Obtain a prediction for a specific user for a particular item 
- Introduce a new user with rating to a rating matrix and make recommendations for them 
- Create a function that will return the top n recommendations for a user 


For this lab, we will be using the famous 1M movie dataset. It contains a collection of user ratings for many different movies. In the last lesson, you were exposed to working with `surprise` datasets. In this lab, you will also go through the process of reading in a dataset into the `surprise` dataset format. To begin with, load the dataset into a Pandas DataFrame. Determine which columns are necessary for your recommendation system and drop any extraneous ones.

In [1]:
import pandas as pd
df = pd.read_csv('./ml-latest-small/ratings.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [2]:
# Drop unnecessary columns
new_df = df.drop(columns='timestamp', axis=1)

It's now time to transform the dataset into something compatible with `surprise`. In order to do this, you're going to need `Reader` and `Dataset` classes. There's a method in `Dataset` specifically for loading dataframes.

In [3]:
from surprise import Reader, Dataset
# read in values as Surprise dataset 
read = Reader()
data= Dataset.load_from_df(new_df, read)

Let's look at how many users and items we have in our dataset. If using neighborhood-based methods, this will help us determine whether or not we should perform user-user or item-item similarity

In [4]:
dataset = data.build_full_trainset()
print('Number of users: ', dataset.n_users, '\n')
print('Number of items: ', dataset.n_items)

Number of users:  610 

Number of items:  9724


## Determine the best model 

Now, compare the different models and see which ones perform best. For consistency sake, use RMSE to evaluate models. Remember to cross-validate! Can you get a model with a higher average RMSE on test data than 0.869?

In [5]:
# importing relevant libraries
from surprise.model_selection import cross_validate
from surprise.prediction_algorithms import SVD
from surprise.prediction_algorithms import KNNWithMeans, KNNBasic, KNNBaseline
from surprise.model_selection import GridSearchCV
import numpy as np

In [6]:
## Perform a gridsearch with SVD
# ⏰ This cell may take several minutes to run
param_grid={
    "n_factors":[20,40,80,100],
    "reg_all":[0.02,0.01,0.05,0.1]
}
gs_cv=GridSearchCV(SVD,param_grid=param_grid,cv=5,measures=["rmse"], n_jobs=-1)

# Fitting the gridsearch
gs_cv.fit(data)

In [7]:
# print out optimal parameters for SVD after GridSearch
print(gs_cv.best_score)
print(gs_cv.best_params)

{'rmse': 0.8694292882587058}
{'rmse': {'n_factors': 100, 'reg_all': 0.05}}


In [8]:
# cross validating with KNNBasic
knn_basic=KNNBasic(sim_options={'name': 'pearson', 'user_based': True})
cv_knn_basic = cross_validate(knn_basic, data, cv=5, n_jobs=-1)

In [9]:
# print out the average RMSE score for the test set
for i in cv_knn_basic.items():
    print(i)
print('Average RMSE for KNNBasic: ', np.mean(cv_knn_basic['test_rmse']))

('test_rmse', array([0.98357162, 0.97516476, 0.9648167 , 0.97600986, 0.96271481]))
('test_mae', array([0.75865829, 0.75471229, 0.74527337, 0.75359885, 0.74471263]))
('fit_time', (2.202075481414795, 2.2854297161102295, 2.2524607181549072, 2.138632297515869, 1.8478333950042725))
('test_time', (4.599544525146484, 4.604726552963257, 4.654469013214111, 4.571232080459595, 2.9201819896698))
Average RMSE for KNNBasic:  0.972455550461973


In [10]:
# cross validating with KNNBaseline
# cross validating with KNNBaseline
knn_baseline=KNNBaseline(sim_options={'name': 'pearson', 'user_based': True})
cv_knn_baseline = cross_validate(knn_baseline, data, cv=5, n_jobs=-1)

In [11]:
# print out the average score for the test set
for i in cv_knn_baseline.items():
    print(i)
print('Average RMSE for KNNBaseline: ', np.mean(cv_knn_baseline['test_rmse']))

('test_rmse', array([0.87872041, 0.87324868, 0.87437221, 0.87462984, 0.87882472]))
('test_mae', array([0.67001133, 0.66815873, 0.6706312 , 0.66708238, 0.66999917]))
('fit_time', (2.810994863510132, 2.978083372116089, 2.8026223182678223, 2.7084734439849854, 1.6004910469055176))
('test_time', (6.383965015411377, 6.414904594421387, 6.439768314361572, 6.391721963882446, 3.5371432304382324))
Average RMSE for KNNBaseline:  0.8759591705470321


Based off these outputs, it seems like the best performing model is the SVD model with `n_factors = 50` and a regularization rate of 0.05. Use that model or if you found one that performs better, feel free to use that to make some predictions.

## Making Recommendations

It's important that the output for the recommendation is interpretable to people. Rather than returning the `movie_id` values, it would be far more valuable to return the actual title of the movie. As a first step, let's read in the movies to a dataframe and take a peek at what information we have about them.

In [12]:
df_movies = pd.read_csv('./ml-latest-small/movies.csv')

In [13]:
df_movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


## Making simple predictions
Just as a reminder, let's look at how you make a prediction for an individual user and item. First, we'll fit the SVD model we had from before.

In [14]:
svd = SVD(n_factors= 50, reg_all=0.05)
svd.fit(dataset)

In [15]:
svd.predict(2, 4)

Prediction(uid=2, iid=4, r_ui=None, est=3.0159802292434787, details={'was_impossible': False})

This prediction value is a tuple and each of the values within it can be accessed by way of indexing. Now let's put our knowledge of recommendation systems to do something interesting: making predictions for a new user!

## Obtaining User Ratings 

It's great that we have working models and everything, but wouldn't it be nice to get to recommendations specifically tailored to your preferences? That's what we'll be doing now. The first step is to create a function that allows us to pick randomly selected movies. The function should present users with a movie and ask them to rate it. If they have not seen the movie, they should be able to skip rating it. 

The function `movie_rater()` should take as parameters: 

* `movie_df`: DataFrame - a dataframe containing the movie ids, name of movie, and genres
* `num`: int - number of ratings
* `genre`: string - a specific genre from which to draw movies

The function returns:
* rating_list : list - a collection of dictionaries in the format of {'userId': int , 'movieId': int , 'rating': float}

#### This function is optional, but fun :) 

In [16]:
def movie_rater(movie_df,num, genre=None):
    userID = 100
    ratingList = []

    while num > 0:
        # filter movies by genre if specified, otherwise select a random movie
        if genre:
            movie = movie_df[movie_df['genres'].str.contains(genre)].sample(1)
        else:
            movie = movie_df.sample(1)
            # get movie title and id
        title = movie['title'].values[0]
        movie_id = movie['movieId'].values[0]
        # ask user to rate movie
        rating = input(f"How would you rate '{title}'? (0.5-5 or type 'no' to skip): ")
        # allow user to skip rating
        if rating.lower() == 'no':
            num -= 1
            continue
        # validate rating input
        try:
            rating_movie = {
                'userId': userID,
                'movieId': int(movie_id),
                'rating': float(rating)
                
            }
            ratingList.append(rating_movie)
            num -= 1
            
        except ValueError:
            print("Invalid rating. Enter a number like 3, 4.5, or type 'no'.\n")

    return ratingList
        

In [17]:
# try out the new function here!
user_ratings = movie_rater(df_movies, 5, genre='Comedy')
user_ratings

[{'userId': 100, 'movieId': 27808, 'rating': 4.0},
 {'userId': 100, 'movieId': 54780, 'rating': 4.0},
 {'userId': 100, 'movieId': 1318, 'rating': 3.0},
 {'userId': 100, 'movieId': 182749, 'rating': 2.0}]

If you're struggling to come up with the above function, you can use this list of user ratings to complete the next segment

In [18]:
user_ratings

[{'userId': 100, 'movieId': 27808, 'rating': 4.0},
 {'userId': 100, 'movieId': 54780, 'rating': 4.0},
 {'userId': 100, 'movieId': 1318, 'rating': 3.0},
 {'userId': 100, 'movieId': 182749, 'rating': 2.0}]

### Making Predictions With the New Ratings
Now that you have new ratings, you can use them to make predictions for this new user. The proper way this should work is:

* add the new ratings to the original ratings DataFrame, read into a `surprise` dataset 
* train a model using the new combined DataFrame
* make predictions for the user
* order those predictions from highest rated to lowest rated
* return the top n recommendations with the text of the actual movie (rather than just the index number) 

In [19]:
## add the new ratings to the original ratings DataFrame
user_ratings = pd.DataFrame(user_ratings)
new_ratings = pd.concat([new_df, user_ratings], ignore_index=True)

In [20]:
# train a model using the new combined DataFrame
new_data = Dataset.load_from_df(new_ratings, read)
trainset = new_data.build_full_trainset()

svd_one = SVD(n_factors=50, reg_all=0.05)
svd_one.fit(trainset)

In [21]:
# make predictions for the user
# you'll probably want to create a list of tuples in the format (movie_id, predicted_score)
rated_movies = user_ratings["movieId"].tolist()

list_of_movies = []

for movie_id in new_df["movieId"].unique():
    if movie_id not in rated_movies:
        pred = svd_one.predict(100, movie_id)
        list_of_movies.append((movie_id, pred.est))

In [23]:
# order the predictions from highest to lowest rated
ranked_movies = sorted(list_of_movies, key=lambda x: x[1], reverse=True)
ranked_movies[:10]

[(898, 4.667668244743667),
 (318, 4.652859300872081),
 (3451, 4.609736058654067),
 (922, 4.5804163850260045),
 (1204, 4.573951149444312),
 (2959, 4.5668584648787425),
 (750, 4.563409041622815),
 (50, 4.56096135780961),
 (58559, 4.549834506568416),
 (177593, 4.549643087951848)]

 For the final component of this challenge, it could be useful to create a function `recommended_movies()` that takes in the parameters:
* `user_ratings`: list - list of tuples formulated as (user_id, movie_id) (should be in order of best to worst for this individual)
* `movie_title_df`: DataFrame 
* `n`: int - number of recommended movies 

The function should use a `for` loop to print out each recommended *n* movies in order from best to worst

In [25]:
# return the top n recommendations using the 
def recommended_movies(ranked_movies, movie_title_df, n):
    
    top_recommendations = ranked_movies[:n]
    
    recommended = []

    for movie in top_recommendations:
        movie_id = movie[0]
        predicted_rating = movie[1]
        
        title = movie_title_df[movie_title_df["movieId"] == movie_id]["title"].values[0]
        
        recommended.append((title, predicted_rating))
        
    return recommended

recommended_movies(ranked_movies,df_movies,5)

[('Philadelphia Story, The (1940)', 4.667668244743667),
 ('Shawshank Redemption, The (1994)', 4.652859300872081),
 ("Guess Who's Coming to Dinner (1967)", 4.609736058654067),
 ('Sunset Blvd. (a.k.a. Sunset Boulevard) (1950)', 4.5804163850260045),
 ('Lawrence of Arabia (1962)', 4.573951149444312)]

## Level Up (Optional)

* Try and chain all of the steps together into one function that asks users for ratings for a certain number of movies, then all of the above steps are performed to return the top $n$ recommendations
* Make a recommender system that only returns items that come from a specified genre

## Summary

In this lab, you got the chance to implement a collaborative filtering model as well as retrieve recommendations from that model. You also got the opportunity to add your own recommendations to the system to get new recommendations for yourself! Next, you will learn how to use Spark to make recommender systems.